# 📦 Projeto de Portfólio: Analytics E-commerce Brasil (Olist)

Este notebook apresenta uma Análise Exploratória de Dados (EDA) avançada sobre o ecossistema de e-commerce da Olist. O objetivo é diagnosticar gargalos operacionais, validar regras de negócio ocultas e mapear o impacto da eficiência logística na satisfação do cliente final.

## 🛠️ FASE 1: ANÁLISE DE SANIDADE E QUALIDADE (DATA PROFILING)

Nesta fase, realizamos o inventário inicial dos dados, mapeando a estrutura das tabelas relacionais, ajustando tipagens corrompidas e auditando o comportamento dos dados nulos.

### Etapa 1.1: Carregamento Dinâmico e Auditoria de Tipagem
* **Objetivo:** Leitura otimizada dos arquivos CSV sem redundância de código e verificação dos tipos primitivos das colunas.
* **Ações:** Execução do loop global de carga e mapeamento de tipos via `.dtypes`.

In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:


# 1. Caminho relativo para a pasta de dados brutos (subindo um nível a partir de 'notebooks')
DATA_RAW_DIR = os.path.join("..", "data", "raw")

# 2. Mapeamento explícito do arquivo para o nome final da variável (Melhor Prática)
# Isso garante controle total sobre o escopo global do seu ambiente de análise
file_mapping = {
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_products_dataset.csv": "products",
    "olist_customers_dataset.csv": "customers",
    "olist_sellers_dataset.csv": "sellers",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "category_translation"
}

print("🔍 Iniciando carga otimizada e desempacotamento dos datasets para EDA...\n")
print(f"📂 Diretório de origem: {os.path.abspath(DATA_RAW_DIR)}\n")

# 3. Loop de leitura e injeção automática no escopo global do Notebook
for filename, var_name in file_mapping.items():
    file_path = os.path.join(DATA_RAW_DIR, filename)
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        globals()[var_name] = df
        print(f"✅ Variável '{var_name}' criada com sucesso!")
        print(f"   📊 Formato: {df.shape[0]:,} linhas × {df.shape[1]} colunas\n")
    else:
        print(f"⚠️ Alerta: O arquivo '{filename}' não foi encontrado na pasta 'data/raw/'.\n")

print("Processo concluído! ")

🔍 Iniciando carga otimizada e desempacotamento dos datasets para EDA...

📂 Diretório de origem: d:\Projetos\Analytics-Ecom-Brasil\data\raw

✅ Variável 'orders' criada com sucesso!
   📊 Formato: 99,441 linhas × 8 colunas

✅ Variável 'order_items' criada com sucesso!
   📊 Formato: 112,650 linhas × 7 colunas

✅ Variável 'products' criada com sucesso!
   📊 Formato: 32,951 linhas × 9 colunas

✅ Variável 'customers' criada com sucesso!
   📊 Formato: 99,441 linhas × 5 colunas

✅ Variável 'sellers' criada com sucesso!
   📊 Formato: 3,095 linhas × 4 colunas

✅ Variável 'payments' criada com sucesso!
   📊 Formato: 103,886 linhas × 5 colunas

✅ Variável 'reviews' criada com sucesso!
   📊 Formato: 99,224 linhas × 7 colunas

✅ Variável 'geolocation' criada com sucesso!
   📊 Formato: 1,000,163 linhas × 5 colunas

✅ Variável 'category_translation' criada com sucesso!
   📊 Formato: 71 linhas × 2 colunas

Processo concluído! 


In [10]:

datasets_to_inspect = [
    "orders", "order_items", "products", "customers", 
    "sellers", "payments", "reviews", "geolocation", "category_translation"
]

print("📊 INICIANDO AUDITORIA DE METADADOS (ESTRUTURA & TIPAGEM) 📊\n")
print("=" * 70)


for var_name in datasets_to_inspect:
    if var_name in globals():
        df = globals()[var_name]
        
        # Cabeçalho do dataset atual
        print(f"📦 DATASET: '{var_name}'")
        print(f"📐 Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
        print("-" * 70)
        
        # Criando uma tabela resumo de metadados para este dataset
        metadata_df = pd.DataFrame({
            'Coluna': df.columns,
            'Tipo de Dado': df.dtypes.values,
            'Registros Preenchidos': df.notnull().sum().values,
            'Nulos (%)': (df.isnull().sum().values / len(df) * 100).round(2)
        })
        
        # Exibe o sumário das colunas formatado
        print(metadata_df.to_string(index=False))
        print("\n" + "=" * 70 + "\n")
        
    else:
        print(f"⚠️ Alerta: A variável '{var_name}' não foi encontrada na memória.")
        print("=" * 70 + "\n")

📊 INICIANDO AUDITORIA DE METADADOS (ESTRUTURA & TIPAGEM) 📊

📦 DATASET: 'orders'
📐 Dimensões: 99,441 linhas × 8 colunas
----------------------------------------------------------------------
                       Coluna Tipo de Dado  Registros Preenchidos  Nulos (%)
                     order_id          str                  99441       0.00
                  customer_id          str                  99441       0.00
                 order_status          str                  99441       0.00
     order_purchase_timestamp          str                  99441       0.00
            order_approved_at          str                  99281       0.16
 order_delivered_carrier_date          str                  97658       1.79
order_delivered_customer_date          str                  96476       2.98
order_estimated_delivery_date          str                  99441       0.00


📦 DATASET: 'order_items'
📐 Dimensões: 112,650 linhas × 7 colunas
--------------------------------------------------

### Etapa 1.2: Investigação de Valores Ausentes (Missing Values)
* **Objetivo:** Mapeamento percentual de nulos em todos os datasets, com foco no rastreamento de nulos estruturais vs. anomalias de sistema.
* **Ações:** Filtro de registros nulos em `order_delivered_customer_date` cruzado com o `order_status`.

## 📊 FASE 2: ANÁLISE UNIVARIADA E ESTATÍSTICA DESCRITIVA

Estudo isolado das métricas de performance, variáveis financeiras e demográficas para compreender a distribuição física dos dados e identificar anomalias matemáticas.

### Etapa 2.1: Distribuição do Tempo Real de Entrega
* **Objetivo:** Calcular e analisar o tempo gasto em dias entre a compra e a entrega efetiva na casa do cliente.
* **Métrica:** `tempo_entrega_dias` plotado via histograma isolado (limite de 100 dias, intervalos de 10 em 10).

### Etapa 2.2: Distribuição do Cumprimento de SLA (Previsão de Entrega)
* **Objetivo:** Analisar o comportamento do erro de previsão (diferença entre a data real de entrega e a estimada prometida ao usuário).
* **Métrica:** `dias_diferenca_estimativa` com linha de referência vertical no ponto zero (limite do prazo).

### Etapa 2.3: Dispersão e Identificação de Outliers de Preço e Frete
* **Objetivo:** Análise quantílica dos valores financeiros de venda e envio para separar o comportamento padrão de desvios abusivos.
* **Ações:** Geração de boxplots isolados para as variáveis `price` e `freight_value`.

## 🧬 FASE 3: ANÁLISE MULTIVARIADA E CORRELAÇÕES

Fase de cruzamento de dados onde unimos tabelas distintas para rastrear interações de causa e efeito, validando ou refutando as hipóteses de negócio levantadas.

### Etapa 3.1: Impacto Logístico na Satisfação do Cliente
* **Objetivo:** Unir a jornada logística ao feedback do usuário para medir a elasticidade da nota e destrinchar o "Paradoxo do SLA".
* **Cruzamento:** `review_score` × `tempo_entrega_dias` × `dias_diferenca_estimativa`.

### Etapa 3.2: Causa Raiz - Categorias Detratoras e Frete Abusivo
* **Objetivo:** Isolar o Top 10 de categorias de produtos que concentram notas baixas (1 e 2) e avaliar a influência do preço e do frete nestes cenários.
* **Cruzamento:** `product_category_name` × `review_score` × `price` × `freight_value`.

## 🧪 FASE 4: VALIDAÇÃO DAS HIPÓTESES (ESTUDO DE CAUSA RAIZ)

Nesta fase focada, confrontamos os dados cruzados com as suspeitas de negócio levantadas. Aqui testamos estatisticamente os motivos reais que guiam o comportamento dos clientes e geram atritos na plataforma.

## 📖 FASE 5: DOCUMENTAÇÃO VIVA, STORYTELLING E RECOMENDAÇÕES

Tradução técnica de todas as hipóteses validadas em estratégias comerciais, planos de ação práticos e correções de engenharia para o comitê executivo da plataforma.